In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Import Libraries, classes, and functions 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler, StandardScaler, RobustScaler 
from sklearn.impute import KNNImputer
from sklearn.feature_selection import mutual_info_regression 
from sklearn.decomposition import PCA, TruncatedSVD  
from sklearn.compose import make_column_transformer 
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold, StratifiedKFold, StratifiedShuffleSplit  
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier  
from sklearn.svm import SVC 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.naive_bayes import GaussianNB 
from sklearn.neural_network import MLPClassifier 
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score 

from xgboost import XGBClassifier 
from lightgbm import LGBMClassifier 

# Set random seed 
SEED = np.random.default_rng().integers(99999)
# SEED = 999999 
from tensorflow.keras.utils import set_random_seed 
set_random_seed = SEED

In [ ]:
# Import training dataset into padas DataFrame & explore 
train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv') 

print(train.head(), '\n', 
train.info(), '\n', 
train.describe(include='all').T, '\n', 
train.nunique() , '\n', 
train.isna().sum() )

In [ ]:
# Import test dataset into padas DataFrame & explore 
test = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv') 

print(test.head(), '\n', 
test.info(), '\n', 
test.describe(include='all').T, '\n', 
test.nunique() , '\n', 
test.isna().sum() )

In [ ]:
# merge test and train datasets into single dataframe for data cleaning and necessary pre-processing 
all = pd.concat([train, test], axis=0) 

print(all.head(), '\n', 
all.info(), '\n', 
all.describe(include='all').T, '\n', 
all.nunique() , '\n', 
all.isna().sum() )

In [ ]:
# drop duplicate rows 
all.drop_duplicates(keep='first', inplace=True) 
print(all.dtypes)

# Feature Engineering 
# split PassengerId column into its component parts (group, and id ) as separate features  
all[['group', 'id']] = all['PassengerId'].str.split(pat='_', expand=True) 

# split Cabin column into its component parts (deck, cabin id, side ) as separate features 
all[['deck', 'cabin_no', 'side']] = all.Cabin.str.split('/', expand=True) 

# split Name column into its component parts (first and last name) as separate features
all[['FName', 'LName']] = all.Name.str.split(" ", expand=True) 
''' The intuition behind splitting name is that family name or first name may potentially 
have some influence on who gets transported into the anomaly '''

# Treatment of High-cardinality columns : columns that have too many categories (eg. upwards of 25 categories) 
''' add value_counts column for high cardinality columns & then drop original col, and treat resulting values as numerical data 
alternatively, transform high_cardinality column data into numbers (ordinal encoding) and treat as numerical data '''

# drop unique value columns  
print(all.columns)

# Subsetting columns for pre-processing with relevant encoders 
''' Since most machine learning models will require all data in numerical form, we will use relevant 
encoders to transform our data accordingly. Numerical columns will not undergo any encoding. 
Columns with ordinal values (ranks, value_counts, high-cardinality columns, etc) will be processed 
using Ordinal Encoder. Columns with categorical values (low-cardinality unique un-ordered values) 
to be OneHotEncoded, but we will first convert them into numerical codes (with Ordinal Encoder) so 
the whole dataset is already numerical for missing value imputation using an Imputation algorithms, 
eg. KNNImputer we have used here. '''

numeric = ['Age', 'group', 'id', 'cabin_no', 'RoomService', 'FoodCourt', 'ShoppingMall','Spa','VRDeck',] 
ordinal = ['FName', 'LName'] 
categorical = ['HomePlanet', 'Destination', 'deck', 'side', 'CryoSleep', 'VIP', 'Transported'] 

# checking unique values in each categorical columns to address 
# any mistakes (such as due to spelling, uppercase-lowercase or similar errors,)
uniq_cats = {col: sorted(all[col].dropna().unique().tolist() ) for col in categorical} 
# print(uniq_cats) 

# Converting target column booleans to corresponding Integer values 
all.Transported = all.Transported.replace({True: 1, False:0})

# Viewing summary statistics for all columns to check for any un-natural (eg. negative values where there should be none, etc) values,
print(all.describe(include='all').T)

# converting into numeric dtypes where numbers are appearing as object datatype 
all[['group', 'id', 'cabin_no']] = all[['group', 'id', 'cabin_no']].apply(pd.to_numeric, axis=0)
print(all.info())

In [ ]:
# dropping columns that have been split into their component values 
# setting original PassengerId column as row index 
all.set_index('PassengerId', inplace=True) 

# dropping Cabin & Name columns that were split previously 
all.drop(['Cabin', 'Name'], axis=1, inplace=True)

# checking all columns in our dataset match with those covered under column groupings
print(len(all.columns), len(numeric + ordinal + categorical) )

# Transforming all NA value types (pandas, numpy NA types, etc) into np.nan type for uniformity 
all = all.fillna(np.nan) 

# Splitting our merged dataset into original train and test sets, 
# where original Test Set had missing ('NA') values in Transported column 
X_train = all[all.Transported.notna()].drop('Transported', axis=1)
y_train = all[all.Transported.notna()].Transported.astype(int) 
X_test = all[all.Transported.isna()].drop('Transported', axis=1)

# deleting the target column name from our categorical columns grouping list 
categorical.remove('Transported')

# checking for data imbalance
print('Mean of target classes: ', y_train.mean())

''' Since this is a binary classification problem, calculating mean value of target column will suffice. 
Here it is 0.503 indicating that the data very well balanced into positive and negative target classes. 
In case we had severely unbalanced data between target classes (say 0.75:0.25), we may have to address 
the imbalance, for example, by bootstrapping the minority class column to match the sample size of the 
majority class, or alternatively sampling from majority class column to match sample size of minority class''' 

In [ ]:
# Using encoders on column groupings to convert data into numerical values where applicable 
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan) 

''' we will fit the encoders to training data and transform both training and test datasets using the 
same patterns learned from training data. This is to avoid leakage (of insights from test data (which 
is not available at the time of training in real world application) into training dataset) '''
X_train[ordinal + categorical] = ordinal_encoder.fit_transform(X_train[ordinal + categorical]) 
X_test[ordinal + categorical] = ordinal_encoder.transform(X_test[ordinal + categorical])

# check dtypes for each column and correct where necessary 
print(X_train.info(), y_train.info(), X_test.info(), sep = '\n\n')
print(X_train.describe(), y_train.describe(), X_test.describe(), sep = '\n\n')

In [ ]:
# Scaling data before using missing value Imputation algorithm 
# calculate min & max values for all categorical features combined 
min_cat = X_train[categorical].min().min()
max_cat = X_train[categorical].max().max()
print('Min-Max spread of values in categorical columns: \n', min_cat, max_cat) 

# Scaling using MinMaxScaler 
mmScaler = MinMaxScaler(feature_range=(min_cat, max_cat)) 
X_train[numeric + ordinal] = mmScaler.fit_transform(X_train[numeric + ordinal]) 
X_test[numeric + ordinal] = mmScaler.transform(X_test[numeric + ordinal]) 

# checking column-wise standard deviation for scaled data 
print('column-wise standard deviation in Training set after min-max scaling')
print(X_train.std(axis=0).T)
print('column-wise standard deviation in Test set after min-max scaling')
print(X_test.std(axis=0).T)

In [ ]:
# Using KNN Imputer to impute missing values 

# ensure all NA values are represented uniformluy as np.nan 
X_train = X_train.fillna(np.nan)
X_test = X_test.fillna(np.nan) 

# Instantiate KNNImputer using default 5 neighbors 
imputer = KNNImputer(n_neighbors=5) 


X_train.loc[:] = imputer.fit_transform(X_train) 
X_test.loc[:] = imputer.transform(X_test) 

# Checking if any missing values still remain 
# print(X_train.isna().sum().T) 
# print(X_test.isna().sum().T) 

In [ ]:
# Calculating correlation (linear) and mutual-info (non-linear) of dataset columns / features with target column 
correlation = pd.concat([X_train, y_train], axis=1).corr().Transported.abs().values[:-1]
mutual_info = mutual_info_regression(X_train, y_train, random_state=SEED) 
associations = {'correlation': correlation, 'mutual_info': mutual_info} 
associations = pd.DataFrame(associations, index=X_train.columns).sort_values(by='correlation', ascending=False)
display(associations.style.background_gradient() )


# Checking summary statistics after missing value imputation 
print('Summary statistics after missing value imputation: ') 
print(X_train.info(), X_train.describe().T, sep='\n') 
print(X_test.info(), X_test.describe().T, sep='\n') 

In [ ]:
# Transforming Categorical features using OneHotEncoder 
print('Before OHE', X_train.shape, X_test.shape, sep='\n')
onehot = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False) 
onehot = make_column_transformer((onehot, categorical), remainder='passthrough')
X_train = onehot.fit_transform(X_train)
X_test = onehot.transform(X_test) 
print('After OHE', X_train.shape, X_test.shape, sep='\n')

# scaler = StandardScaler(with_mean=True)
scaler = RobustScaler(with_centering=True)
X_train = scaler.fit_transform(X_train) 
X_test = scaler.transform(X_test) 

# Retaining full training set for training model to be used for submission  
X_train_full, y_train_full = X_train.copy(), y_train.copy()

# Creating training and validation split (including corresponding indices from the base set) to evaluate model performances 
X_train, X_val, y_train, y_val, tts_tr, tts_val = train_test_split(X_train_full, y_train_full, np.arange(X_train_full.shape[0]), 
                                                                   test_size=0.20, stratify=y_train_full, random_state=SEED) 
print([i.shape[0] for i in [X_train, X_val, y_train, y_val, tts_tr, tts_val] ]) 
print('SEED: ', SEED)

In [ ]:
 pred_dict = dict() # to save separate predictions basis shuffled splits of training data 
n_splits = 5 # no of shuffled splits of data we will try 

In [ ]:
sss = StratifiedShuffleSplit(n_splits=n_splits, test_size=2, random_state=SEED, ) 

for i, (tr_ids, ts_ids) in enumerate(sss.split(X_train_full, y_train_full)): 
    # training on shuffled dataset with 0.2 validation_fraction 
    model = GradientBoostingClassifier(max_depth=7, max_features='sqrt', n_estimators=600, n_iter_no_change=50, tol=0, subsample=0.75, learning_rate=0.05, validation_fraction=0.2, verbose=0, random_state=SEED+i )
    model = model.fit(X_train_full[tr_ids], y_train_full[tr_ids] ) 
    # Checking OOB scores 
    raw_ests = model.n_estimators_ 
    opt_ests = np.argmax(np.cumsum(model.oob_improvement_)) + 1 
    estop_ests = raw_ests - model.n_iter_no_change 
#     print(opt_ests, raw_ests, sep=' / ' )
    print('Peak oob cum. improv: ', np.max(np.cumsum(model.oob_improvement_)), np.cumsum(model.oob_improvement_)[opt_ests-1] )
    print('Before n_est tuning', i, np.sum(model.oob_improvement_), sep=': ')
#     print(i, 'log_loss', log_loss(y_train_full[ts_ids], model.predict_proba(X_train_full[ts_ids])[:,1])) 
#     print(i, 'accuracy', model.score(X_train_full[ts_ids], y_train_full[ts_ids]), '\n') 
    
    ''' To refit the model using n_estimators corresponding to peak cum. improvement, use n_estimators=opt_ests '''
    
    # Refit the model using n_estimators = raw_ests - early stopping 
    model = GradientBoostingClassifier(max_depth=7, max_features='sqrt', n_estimators=estop_ests, n_iter_no_change=50, tol=0, subsample=0.75, learning_rate=0.05, validation_fraction=0.2, verbose=0, random_state=SEED+i) 
    model = model.fit(X_train_full[tr_ids], y_train_full[tr_ids]) 
    # Checking OOB scores 
    opt_ests = np.argmax(np.cumsum(model.oob_improvement_)) + 1 
#     print(opt_ests, model.n_estimators_, sep=' / ' )
    print('Peak oob cum. improv: ', np.max(np.cumsum(model.oob_improvement_)), np.cumsum(model.oob_improvement_)[opt_ests-1] )
    print('After n_est tuning', i, np.sum(model.oob_improvement_), sep=': ')
#     print(i, 'log_loss', log_loss(y_train_full[ts_ids], model.predict_proba(X_train_full[ts_ids])[:,1])) 
#     print(i, 'accuracy', model.score(X_train_full[ts_ids], y_train_full[ts_ids]), '\n') 

    # predict on test set using model trained on StratifiedShuffleSplit of training data with 0.2 validation_fraction 
    pred_dict['gbc_'+str(i)] = model.predict_proba(X_test)[:, 1] 
    
predict_frame = pd.DataFrame(pred_dict)
predict_frame['gb_avg'] = predict_frame.iloc[:,-n_splits:].mean(axis=1) # soft voting gives better results than hard voting 
# predict_frame['gb_avg'] = predict_frame.median(axis=1) # hard voting  
print(predict_frame.describe()) 

y_test_pred = np.round(predict_frame['gb_avg'].values)  
# y_test_pred = np.round(predict_frame[2].values)  
print(y_test_pred[:5]) 
print(np.mean(y_test_pred)) 

submission = pd.DataFrame(dict(PassengerId = test.PassengerId, Transported = y_test_pred)) 
submission.Transported = submission.Transported.astype('bool')
submission.to_csv('submission.csv', index=False)
submission.head() 

print('SEED:', SEED) # to see THE random seed used in this run 
print ( ((predict_frame > 0.45) & (predict_frame < 0.55)).sum() )